# Network Expansion
## Model 1 - Deterministic Baseline

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.classes import DistributionNetwork
from src.classes import Substation
from src.solver import solve_network
from src.solver import print_results

![Topology example](figures/Model1_Initial.png)  
*Example topology of a distribution system evaluated in this notebook.*

### 1- Define the Distribution Network

In [2]:
NODES = [f"N{i}" for i in range(1,14)] # Listing system nodes: 'N1','N2', ..., 'N13'
LOADS = [f'D{i}' for i in range(1,11)] # Listing system loads: 'D1', 'D2', ..., 'D10'
# NOTE: Candidate nodes (ones with potential substations) are added separately using add_candidate_substations method.

# Initial (online) substations
# Substation S1 at node N4, with capacity 40, connected to nodes N3, N5, N9, with capacity reinforcement cost of 200 and feeder line cost of 50.
S1 = Substation("S1", "N4", 40, ["N3", "N5", "N9"], r_cost = 200, edge_cost = 50)
SUBSTATIONS = [S1] # Substations to pass to a class

line_cost = 50 # Default cost of building network lines (can be different from substation feeder lines)

load_capacity = {'D1': 6,
                 'D2': 3,
                 'D3': 2,
                 'D4': 5,
                 'D5': 3,
                 'D6': 2,
                 'D7': 3,
                 'D8': 5,
                 'D9': 4,
                 'D10': 6}

Mapping distribution lines and system loads (nodal locations)

In [3]:
# Loads
loads_locations = {
    "D1": "N1",
    "D2": "N2",
    "D3": "N3",
    "D4": "N6",
    "D5": "N7",
    "D6": "N8",
    "D7": "N9",
    "D8": "N11",
    "D9": "N12",
    "D10": "N13",
}

# Distribution lines
nodes_connected = {
    "N1": ["N2"],
    "N2": ["N1", "N3"],
    "N3": ["N2", "N4"],
    "N4": ["N3", "N5", "N9"],
    "N5": ["N4", "N6"],
    "N6": ["N5","N7", "N8"],
    "N7": ["N6"],
    "N8": ["N6"],
    "N9": ["N4", "N10"],
    "N10": ["N9", "N11", "N13"],
    "N11": ["N10", "N12"],
    "N12": ["N11"],
    "N13": ["N10"]
}

Instancing a pre-defined Distribution Network

In [4]:
DistributionNetwork = DistributionNetwork(NODES, 
                                          LOADS, 
                                          SUBSTATIONS, 
                                          load_capacity,
                                          nodes_connected,
                                          loads_locations,
                                          line_cost)

Adding *candidate* substations for expansion (with potential connection lines).

In [5]:
capacity = 15
s_cost = 100        # Cost of substation activation
l_cost = line_cost  # Cost of connecting a substation feeder line (set equal to other network lines = 50)
r_cost = 200        # Cost of capacity reinforcement (set equal to existing substation)

S2 = Substation("S2", "N14", capacity, ["N2"], r_cost, edge_cost = l_cost, fix_cost = s_cost) # Potential substation S2 at node N14, with potential connection to node N2
S3 = Substation("S3", "N15", capacity, ["N6"], r_cost, edge_cost = l_cost, fix_cost = s_cost)
S4 = Substation("S4", "N16", capacity, ["N11", "N13"], r_cost, edge_cost = l_cost, fix_cost = s_cost)
DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 2 - Define a Network Expansion optimization problem
Based on the radial operation of distribution systems, we model a baseline network expansion optimization problem. 
Network expansion here focuses on reinforcements in substations and feeder lines. Concretely, the network can be reinforced 
in three ways:
1. Activating and connecting **new substations** to supply power in areas of high demand
2. Adding, connecting new **feeder lines** (reconfiguring network) for power flow optimization
3. Reinforcing existing substations by incresing their capacity

Both actions 1. and 2. can cause major network reconfiguration, as distribution networks operate radially, meaning 
each demand can be supplied by one, and only one, substation.  

Full `solve_network` function can be found in `src/solver.py`

1. **Parameters**

- $L_d:$ Power consumption (demand) of load $d$.
- $P^S_s:$ Capacity of substation $s$.
- $C^{S}_s:$ Cost of activating substation $s$.
- $C^{L}_{(i,j)}:$ Cost of connecting line $(i,j)$.
- $C^R:$ Cost of capacity reinforcement.
- $R:$ Size of a single capacity reinforcement.
- $B:$ Annual budget.
- $dr:$ Discount rate.
- $Y:$ Year (used only in Model 1 because of a singe-period optimization)
- $M:$ Big-M used to constraint $f$ (power flow). Equal to total system demand.

2. **Decision variables**
- $w_s \in \{0,1\}$ Substation Activation Variable. Indicates whether substation $s$ is activated.
- $y_{i,s} \in \{0,1\}$ Node Assignment Variable. Indicates whether node $i$ is served by substation $s$.
- $x_{i,j,s} \in \{0,1\}$ Arc Usage Variable. Indicates whether arc $(i,j)$ is used by flows from substation $s$. 
- $f_{s,i,j} \geq 0 $ Power Flow Variable. Represents the amount of power flow from substation $s$ travelling along directed arc $(i,j)$.
- $r_s \geq 0 $ Substation Supply. Represents total demand supplied by substation $s$.
- $z_s \in \Z$ Capacity Reinforcement Variable. Indicates how many times substation $s$ upgraded its maximum capacity.

3. **Constraints**  
- **(1) - Power Balance**  
  All power flows sum up to total system demand (are 100% covered).
  $$ \sum\limits_{s \in \mathcal{S}} r_s = \sum\limits_{d \in \mathcal{D}} L_d$$

- **(2) - Node Assignment**  
  Node $n$ is assigned to substation $s$, only if substation $s$ is active.
  $$ y_{n,s} \le w_s \quad \forall n,\forall s $$

  **(2a) Non-substation node**  
  Non-substation node must be assigned to one, and only one, substation.
  $$ \sum_{s \in S} y_{n,s} = 1 \quad \forall n \in \mathcal{N} \setminus \mathcal{N}^S $$
  
  **(2b) Substation node**  
  Substation node is assigned to its own substation (if activated) (1).  
  Substation nodes cannot be assigned to other substations (2).
  $$
  \begin{align}
    y_{n,s_n} &= w_{s_n} & \forall n \in \mathcal{N}^S ,\\
    y_{n,s} &= 0 & \forall n \in \mathcal{N}^S, \forall s \in \mathcal{S} \setminus \{s_n\}
  \end{align}
  $$
  Where: $s_n: \mathcal{N}^S \longrightarrow \mathcal{S}$, $s_n$ is the substation at node $n$.

- **(3) - Substation Supply**  
  Power flow from a substation is equal to demands supplied by this substation.
  $$ r_s = \sum\limits_{n \in \mathcal{N}}L_{d_n} \cdot y_{n,s} \quad \forall s \in \mathcal{S}$$
  Where: $d_n: \mathcal{N} \longrightarrow \mathcal{D}$, $d_n$ is the load at node $n$.

- **(4) - Capacity Constraint**  
  Power flow from a substation is lower or equal to substation's max capacity + potential capacity 
  reinforcements. Equal to zero if substation is inactive.
  $$ r_s \leq (P^S_s + R \cdot z_s) w_s \quad \forall s \in \mathcal{S}$$

- **(5) - Substation Feeder Line Activation**  
  If a substation is activated, at least one feeder line is always used by flows from it.
  $$w_s \leq \sum\limits_{(i,j)\in \mathcal{A}} x_{(i,j),s} \quad \forall s \in \mathcal{S} $$

- **(6) - Flow conservation**  
  *Substation nodes* (roots): The net outgoing flow equals the substation's supply (1).  
  *Non-substation nodes*: The net flow satisfies the node's demand if it is assigned to the substation (2).
  \begin{align}
    \sum_{(n,j)\in \mathcal{A}} f_{s,(n,j)} - \sum_{(i,n)\in \mathcal{A}} f_{s,(i,n)} 
    &= r_s 
    & \forall s \in S,\; n \in \mathcal{N}^S,\; s = s_n, \\[2mm]
    \sum_{(n,j)\in \mathcal{A}} f_{s,(n,j)} - \sum_{(i,n)\in \mathcal{A}} f_{s,(i,n)} 
    &= -\, L_{d_n} \, y_{n,s} 
    & \forall s \in S,\; n \in \mathcal{N} \setminus \mathcal{N}^S
  \end{align}

- **(7) - Flow only if arc is assigned**  
  Flow through arc is costrained by total system demand. Equal to zero if arc is inactive.
  \begin{align}
    f_{s,(i,j)} &\le M \, x_{(i,j),s} 
    & \forall s \in S, \; \forall (i,j) \in \mathcal{A}
  \end{align}

- **(8) - Distribution Network Radiality**  
Each *non-substation node* assigned to substation must have exactly one incoming arc $(j,n)$, 
ensuring it has a unique parent in the tree (1).  
*Substation nodes* have no incoming arcs, so they serve as the root of the tree (2).  
*Substation node* assigned to the substation $s$ forbids other substations from using outgoing arcs from $n_s$ (3).
  \begin{align}
    \sum_{(j,n)\in \mathcal{A}} x_{(j,n),s} &= y_{n,s} 
    & \forall s \in S, \; \forall n \in \mathcal{N} \setminus \mathcal{N}^S, \\[2mm]
    \sum_{(j,n_s)\in \mathcal{A}} x_{(j,n_s),s} &= 0 
    & \forall s \in S, \\[2mm]
    \sum_{\substack{s_x \in \mathcal{S} \\ s_x \neq s}} x_{(n_s,j),s_x} &\leq 1 - y_{n_s,s} & \forall s \in \mathcal{S}, \; \forall (n_s,j) \in \mathcal{A}
  \end{align}
Where: $n_s: \mathcal{S} \longrightarrow \mathcal{N}^S$, $n_s$ is the node where substation $s$ is located.

- **(9) - Tree size**  
For each substation $s$, the number of arcs in its distribution tree equals the number of 
nodes assigned to it minus one if the substation is active.  
This ensures that each substation’s network forms a connected tree without extra links.
  \begin{align}
    \sum_{(i,j)\in \mathcal{A}} x_{(i,j),s} &= \sum_{n \in \mathcal{N}} y_{n,s} - w_s
    & \forall s \in S
  \end{align}

- **(10) - Initial Constraints**  
  Existing (connected) substations are active.
  $$ w_s = 1 \quad \forall s \in \mathcal{S}_0$$

- **(11) - Annual Budget**  
  Total cost (substations, lines and capacity reinforcements) cannot exceed annual budget.
  \begin{align}
    \sum_{s \in \mathcal{S}} C^S_s \, w_s + 
    \sum_{s \in \mathcal{S}} \sum_{(i,j)\in \mathcal{A}} C^L_{i,j} \, x_{(i,j),s} + 
    \sum_{s \in \mathcal{S}} C^R_s \, z_s
    &\le B
  \end{align}


4. **Model. Objective function**  
$$\min \quad substation\ cost + feeder\ cost + capacity\ cost$$
where: 
- $substation\ cost = \sum\limits_{s \in S} C^S_s w_s$ is a total cost of activating new substations
- $feeder\ cost = \sum\limits_{s\in S}\sum\limits_{(i,j)\in A}^{}C^L_{(i,j)} x_{(i,j),s}$ is a total cost of connecting new feeder lines
- $capacity\ cost = \sum\limits_{s \in S} C^R z_s$ is a total cost of increasing capacity in existing substations

For simplification purposes, the model does not take into account the capacity or susceptance of distribution lines.

### 3 - Solve the optimization problem  
For Model 1, it is assumed that EV and heat-pump deployment cause an increase in demand at a constant rate - **increasing demand** in each load node uniformly **by %6** annualy. The model is simplified by treating this demand as a peak demand, which is supplied at all times. This ultimately satisfies the 99% load-reliability constraint.

In [6]:
R = 10      # Size of a single capacity reinforcement
B = 550     # Annual budget (nominal)
B = 10 * B  # So the total budget is 10 * 550

results = {} # Storing yearly results
system_demand = {} # Also storing system demand (part of deterministic input)
system_demand[0] = list(DistributionNetwork.load_capacity.values()) # Initial demand

demand_rate = 0.06 # Yearly demand growth is 6%
# So in year 10, the demand is 1.06^10 * demand

# Increase demand
for load, demand in DistributionNetwork.load_capacity.items():
    DistributionNetwork.load_capacity[load] = (1 + demand_rate)**10 * demand

system_demand[10] = list(DistributionNetwork.load_capacity.values()) # Storing new total system demand

solution = solve_network(DistributionNetwork, R, B, OutputFlag=1) # Solve
print_results(DistributionNetwork, solution, detailed=True)
DistributionNetwork.update_initial_conditions(solution['w'], solution['x'], solution['z'], R)  # Do this just to display updates.

Set parameter Username
Set parameter LicenseID to value 2706854
Academic license - for non-commercial use only - expires 2026-09-10


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-9300H CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 346 rows, 332 columns and 1309 nonzeros
Model fingerprint: 0x30ca5b1f
Model has 4 quadratic constraints
Variable types: 132 continuous, 200 integer (196 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  QMatrix range    [1e+01, 1e+01]
  QLMatrix range   [1e+00, 4e+01]
  Objective range  [5e+01, 2e+02]
  Bounds range     [1e+00, 7e+01]
  RHS range        [1e+00, 6e+03]
Presolve removed 285 rows and 280 columns
Presolve time: 0.03s
Presolved: 61 rows, 52 columns, 199 nonzeros
Variable types: 16 continuous, 36 integer (32 binary)
Found heuristic solution: objective 800.0000000
Found heuristic solution: objective 750.0000000

Root relaxation: objective 2.795910e+02, 36 itera

### 4 - Results, plots, short insights

In [7]:
results_df = pd.DataFrame({
    "System Demand": round(sum(system_demand[10]), 2),
    "System Supply": round(sum(solution['r'].values()), 2),
    "System Capacity": sum(solution['P'].values()),
    "Substations Active": [[s for s,w in solution['w'].items() if w==1]],
    "Cost": round(solution['objective'], 2)
})

results_df

,System Demand,System Supply,System Capacity,Substations Active,Cost
0,69.84,69.84,80.0,"[1, 2, 4]",500.0


In [8]:
# plt.figure(figsize=(8, 5))
# plt.plot(results_df.index, results_df['Cumulative D. Cost'], marker='o', markersize=4, linewidth=2, color='tab:orange')
# plt.xticks(results_df.index)
# plt.xlabel("Year")
# plt.ylabel("Cumulative Discounted Cost")
# plt.title("Cumulative Cost over Years")
# plt.grid()
# plt.tight_layout()
# plt.show()

# print("Total discounted cost:", round(results_df.loc[10]['Cumulative D. Cost'], 2))

![Final example](figures/Model1_Final.png)  
*Illustration of Year 10 optimization results.*